# Bronze ingestion: Delta change data feed

For an upstream **Delta table** rather than files. Reading its change feed gives
every insert, update and delete as a row, which is what the SCD builds in silver
need. The source table must have CDF enabled:

```sql
ALTER TABLE <source> SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
```

`startingVersion 0` replays from the beginning on the first run; the checkpoint
takes over from there.

Delete this notebook (and its task in `resources/bronze_ingestion.yml`) if the
project has no Delta source.

In [0]:
from pyspark.sql import functions as F

In [0]:
# Per-source constants: the only lines that change when you copy this notebook.
SOURCE_TABLE_NAME = "items"   # table in <prefix>_ingest.<env>
TABLE_NAME = "items_raw"      # target table in the bronze schema
SCHEMA = "bronze"

In [0]:
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV = configs.get("env", "dev")
CATALOG_PREFIX = configs.get("catalog_prefix", "rearc")
INITIAL_RUN = configs.get("initial_run", "False").lower() == "true"

CATALOG = f"{CATALOG_PREFIX}_{ENV}"
INGEST_CATALOG = f"{CATALOG_PREFIX}_ingest"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

source_table = f"{INGEST_CATALOG}.{ENV}.{SOURCE_TABLE_NAME}"
target_table = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
checkpoint_path = f"{CHECKPOINT_BASE}/{TABLE_NAME}/"

print(f"{source_table} -> {target_table} (checkpoint: {checkpoint_path})")

In [0]:
df = (
    spark.readStream.option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .table(source_table)
    .withColumn("_ingested_at", F.current_timestamp())
)

In [0]:
query = (
    df.writeStream.trigger(availableNow=True)
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .toTable(target_table)
)

query.awaitTermination()
print(f"{target_table}: {spark.table(target_table).count():,} rows")

In [0]:
if INITIAL_RUN:
    # keep a longer transaction log than the 30-day default, for debugging
    spark.sql(f"ALTER TABLE {target_table} SET TBLPROPERTIES ('delta.logRetentionDuration' = 'interval 90 days')")
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY (_ingested_at)")